# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Gonablitz/intern-assignment/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

Refresh / Content Opportunity Scoring-It involves building a decision matrix and or ranking system based on structured page/keyword performance features.
I have a good background with Python and decision tree models makes crafting a clear scoring engine, and assigning reason codes/action queues, straightforward and impactful.

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


Refresh / Content Opportunity Scoring - It involves building a decision matrix or ranking system based on structured page/keyword performance features. Your background with Python and decision tree models makes crafting a clear scoring engine ,and assigning reason codes/action queues, straightforward and impactful.

## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


False Positive ;Recommending a refresh on a page that didn't need it:

Direct Cost: Wasted team hours and editorial budget eg; spending $200–$500+ per article rewrite on content that was already performing fine or temporarily fluctuating due to seasonality.

Risk:Inadvertently ruining a well-performing page's current organic ranking by altering core copy that Google already favored.

False Negative ;Failing to flag a decaying high-value page:

Direct Cost: Unnoticed organic traffic loss leading directly to lost leads, conversions, or revenue over weeks or months.

Recovery Cost: Once a page drops significantly off Page 1, re-gaining those positions often costs much more time and backlinks than maintaining a page before it falls out of the top rankings.

The Actors: SEO Strategists, Content Managers, and Copywriters.

The Action: They take the top-ranked pages from the generated review queue, review the attached reason codes  and execute specific content updates—such as updating meta titles/descriptions, refreshing outdated statistics, or expanding thin content sections.

The Decision: Deciding which existing pages or keywords should be prioritized for content refreshes or SEO re-optimization versus which ones should be left alone or created from scratch.

How ML helps: Instead of relying on manual spot-checks or static rules an ML model/scoring system evaluates multi-dimensional signals—such as historical position degradation, CTR gap relative to ranking position, impressions, and search intent value—to output a dynamically ranked priority queue.

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


In [16]:
from datasets import load_dataset
import pandas as pd
import numpy as np
from huggingface_hub import login # Import login function

print("Streaming sample data from FlyRank Hugging Face dataset...")

# Login to Hugging Face Hub
login()

ds = load_dataset("FlyRank/internship-warehouse", "fact_content_daily_performance", streaming=True, split="train")


sample_rows = list(ds.take(10000))
df = pd.DataFrame(sample_rows)

print(f"Sample loaded successfully. Shape: {df.shape}")
print(f"Columns available: {list(df.columns)}")


impressions_col = 'impressions' if 'impressions' in df.columns else 'gsc_impressions'

if 'ctr' not in df.columns and 'click_through_rate' not in df.columns:
    df['gsc_ctr'] = df['gsc_clicks'] / df['gsc_impressions']
    df['gsc_ctr'] = df['gsc_ctr'].fillna(0) # Handle division by zero (0 impressions)
    ctr_col = 'gsc_ctr'
else:
    ctr_col = 'ctr' if 'ctr' in df.columns else 'click_through_rate'

position_col = 'position' if 'position' in df.columns else 'gsc_avg_position'

total_records = len(df)

# 2. High-Impression, Low-CTR Opportunity Pool
high_opp = df[(df[impressions_col] >= df[impressions_col].quantile(0.50)) & (df[ctr_col] < 0.03)]
high_opp_count = len(high_opp)
pct_opportunity = (high_opp_count / total_records) * 100

# 3. CTR Decay Ratio (Positions 1-3 vs 4-10)
top3_ctr = df[df[position_col] <= 3][ctr_col].mean()
pos4_10_ctr = df[(df[position_col] > 3) & (df[position_col] <= 10)][ctr_col].mean()
ctr_multiplier = top3_ctr / pos4_10_ctr if pos4_10_ctr > 0 else 0.0

print("\n--- Section 3: Supporting Data Numbers ---")
print(f"1. Analyzed Sample Volume: {total_records:,} daily performance records")
print(f"2. Refresh Opportunity Pool: {high_opp_count:,} records ({pct_opportunity:.1f}% of total) with high impressions but <3% CTR")
print(f"3. Top 3 CTR Advantage: {ctr_multiplier:.1f}x higher CTR than Positions 4-10 ({top3_ctr*100:.1f}% vs {pos4_10_ctr*100:.1f}%)")

Streaming sample data from FlyRank Hugging Face dataset...


Resolving data files:   0%|          | 0/18 [00:00<?, ?it/s]

Sample loaded successfully. Shape: (10000, 30)
Columns available: ['report_date', 'client_hash_id', 'content_hash_id', 'client_has_gsc', 'client_has_ga4', 'gsc_data_available', 'ga4_data_available', 'gsc_impressions', 'gsc_clicks', 'gsc_sum_position', 'gsc_avg_position', 'ga4_pageviews', 'ga4_sessions', 'ga4_users', 'ga4_engaged_sessions', 'ga4_total_engagement_sec', 'sessions_organic', 'sessions_direct', 'sessions_referral', 'sessions_social', 'sessions_paid', 'sessions_ai', 'ai_chatgpt', 'ai_perplexity', 'ai_gemini', 'ai_copilot', 'ai_claude', 'ai_meta', 'ai_other', 'scroll_events']

--- Section 3: Supporting Data Numbers ---
1. Analyzed Sample Volume: 10,000 daily performance records
2. Refresh Opportunity Pool: 4,679 records (46.8% of total) with high impressions but <3% CTR
3. Top 3 CTR Advantage: 3.3x higher CTR than Positions 4-10 (4.8% vs 1.5%)


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [ ]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


What My Work CAN Claim (Observed, Directional & Decision-Support)
Observed Performance Patterns: Quantify past performance metrics (such as impression volume, ranking position distribution, and click-through rates) based on historical GSC and GA4 daily data.
Directional Opportunity Identification: Identify content items that exhibit significant gaps between impression reach and click conversion relative to position benchmarks, flagging them as high-potential candidates for content or meta optimization.Prioritized Decision-Support Queue: Output a dynamically ranked queue of content candidates accompanied by explicit reason codes, providing content teams with data-backed guidance on where to focus editorial effort.What My Work NEVER Will Claim (Causal Proof & Algorithmic Predictions)No Causal Proof:
The model identifies statistical associations and relative opportunity gaps—it cannot prove causality. A drop in traffic or CTR might be driven by shifting user search intent, external macroeconomic trends, competitor publishing volume, or seasonality, rather than internal content decay.No "Predicting Google": The model does not attempt to reverse-engineer or predict Google's search algorithms, indexing logic, or future ranking positions.No Guaranteed Traffic Outcomes: The model cannot guarantee that updating or refreshing a flagged page will result in a specific rank increase, traffic boost, or revenue gain post-implementation.Summary TableCategoryWhat the Model Does / SaysWhat the Model Avoids / Never ClaimsData ScopeDescribes observed historical metrics across GSC & GA4 performance.Does not assume unobserved variables or missing analytics context.Modeling GoalProvides a directional, weighted opportunity score for task prioritization.Does not claim deterministic guarantees on ranking movements.Business ValueImproves resource allocation efficiency for editorial and SEO teams.Does not claim causal proof for organic traffic fluctuations.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.